# Model Context Protocol (MCP) & LangGraph Step-by-Step Walkthrough

This notebook provides a complete **step-by-step walkthrough** of the candidate evaluation and ranking pipeline. 

Instead of spawning servers as subprocesses inside this process, this notebook assumes you are running the MCP servers independently in separate terminal/command prompt windows over **SSE (Server-Sent Events) HTTP transport**.

### How to run the MCP servers in separate command prompts:
Before executing this notebook, open two separate terminal windows in your project directory and run:

1. **Terminal 1 (Filesystem MCP Server - Port 8001):**
   ```powershell
   .venv\Scripts\python.exe mcp_servers/filesystem_mcp_server.py sse 8001
   ```

2. **Terminal 2 (Candidate Database MCP Server - Port 8002):**
   ```powershell
   .venv\Scripts\python.exe mcp_servers/db_mcp_server.py sse 8002
   ```

Let's get started!

## 1. Setup & Import Dependencies

We import the necessary LangGraph, MCP Client (SSE transport), and support libraries.

In [1]:
import os
import sys
import asyncio
import json
import re
import time
from typing import TypedDict, List, Dict, Any
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print(f"Python interpreter: {sys.executable}")
print(f"Project workspace: {os.getcwd()}")
print(f"OpenRouter API key loaded: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No (using keyword mock fallback)'}")

Python interpreter: d:\Github\MCP-integration\.venv\Scripts\python.exe
Project workspace: d:\Github\MCP-integration
OpenRouter API key loaded: Yes


## 2. Define Server Endpoints

We define the SSE URLs for the Filesystem and Candidate Database servers running on your system.

In [2]:
FS_SERVER_URL = "http://localhost:8001/sse"
DB_SERVER_URL = "http://localhost:8002/sse"

RESUMES_DIR = "./resumes"
REPORTS_DIR = "./reports"
os.makedirs(REPORTS_DIR, exist_ok=True)
print(f"Filesystem MCP URL: {FS_SERVER_URL}")
print(f"Database MCP URL: {DB_SERVER_URL}")
print("Resumes list:", os.listdir(RESUMES_DIR))

Filesystem MCP URL: http://localhost:8001/sse
Database MCP URL: http://localhost:8002/sse
Resumes list: ['Alice_Smith_Resume.pdf', 'Bob_Jones_Resume.pdf', 'Charlie_Brown_Resume.pdf', 'David_Miller_Resume.pdf', 'Resume.pdf']


## 3. Handshake and Discovery

Let's connect to both SSE servers, initialize client sessions, and list their exposed tools and resources.

In [4]:
from mcp import ClientSession
from mcp.client.sse import sse_client

print("Connecting to SSE servers...")
async with sse_client(FS_SERVER_URL) as (fs_read, fs_write), \
           sse_client(DB_SERVER_URL) as (db_read, db_write):
           
    async with ClientSession(fs_read, fs_write) as fs_session, \
               ClientSession(db_read, db_write) as db_session:
               
        await fs_session.initialize()
        await db_session.initialize()
        print("Connections initialized successfully!\n")
        
        print("--- Filesystem MCP Server Tools ---")
        fs_tools = await fs_session.list_tools()
        for t in fs_tools.tools:
            print(f"- {t.name}: {t.description}")
            
        print("\n--- Database MCP Server Tools ---")
        db_tools = await db_session.list_tools()
        for t in db_tools.tools:
            print(f"- {t.name}: {t.description}")

Connecting to SSE servers...
Connections initialized successfully!

--- Filesystem MCP Server Tools ---
- list_directory: 
List all files in the given directory path.
Enforces security boundary checks. Returns a JSON-serialized list of filenames.

- read_file: 
Read the contents of a file. Supports text files and PDF file parsing.
Enforces security boundary checks.

- write_file: 
Write content to a file at the specified path. Creates directories if necessary.
Enforces security boundary checks.

- watch_directory: 
Monitor the specified directory for new files.
First Call: Starts the polling watcher.
Subsequent Calls: Returns any newly added files since the last check.

- batch_process: 
Process multiple files in parallel and return a JSON-serialized mapping of path to file contents.
Enforces security boundary checks on each file path.


--- Database MCP Server Tools ---
- get_candidate_profile: 
Retrieve detailed candidate profile metadata from the database.
Includes verification data

## 4. Direct API Calls (Tool Executions)

Before assembling the LangGraph agent, let's call filesystem and database tools directly to verify functionality.

In [5]:
async with sse_client(FS_SERVER_URL) as (fs_read, fs_write), \
           sse_client(DB_SERVER_URL) as (db_read, db_write):
           
    async with ClientSession(fs_read, fs_write) as fs_session, \
               ClientSession(db_read, db_write) as db_session:
               
        await fs_session.initialize()
        await db_session.initialize()
        
        # 1. Parse a PDF resume using the Filesystem server
        res = await fs_session.call_tool("read_file", arguments={"path": "./resumes/Alice_Smith_Resume.pdf"})
        content = res.content[0].text if hasattr(res, "content") else str(res)
        print("--- parsed PDF Resume snippet ---")
        print(content[:250].strip())
        
        # 2. Get a candidate profile using the Database server
        db_res = await db_session.call_tool("get_candidate_profile", arguments={"name": "Alice Smith"})
        profile = db_res.content[0].text if hasattr(db_res, "content") else str(db_res)
        print("\n--- Database Profile Metadata ---")
        print(profile)

--- parsed PDF Resume snippet ---
--- Page 1 ---
RESUME: ALICE SMITH
Alice Smith - Senior Backend Engineer
Email: alice@example.com | Tech Stack: Python, LangChain, LangGraph, AWS, Docker,
PostgreSQL.
Experience:
- 5 years building backend architectures and microservices.
- Developed

--- Database Profile Metadata ---
{
  "name": "Alice Smith",
  "experience": "5 years as Senior Software Engineer",
  "background_check": "Passed",
  "expected_salary": "$120,000",
  "previous_company": "TechCorp",
  "certifications": [
    "AWS Solutions Architect",
    "Certified Scrum Master"
  ],
  "notes": "Highly recommended for cloud native roles."
}


## 5. Define LangGraph State & Globals

We define the structure of our agent's state and create global reference points so that our graph nodes can access the active sessions.

In [6]:
class AgentState(TypedDict):
    job_description: str
    resumes_dir: str
    resume_paths: List[str]
    resume_contents: Dict[str, str]            # path -> text
    candidate_profiles: Dict[str, Dict[str, Any]] # path -> HR database profile
    match_reports: List[Dict[str, Any]]        # list of match evaluation dicts
    final_report: str

# Global references for sessions inside node calls
GLOBAL_FS_SESSION = None
GLOBAL_DB_SESSION = None

## 6. Define Scoring & Evaluation Logic

We write helper functions to parse tool results, extract JSON outputs from LLM responses, and evaluate candidates (using OpenRouter/OpenAI API or a local keyword fallback).

In [7]:
def parse_mcp_result(result) -> Any:
    """Safely extracts and parses JSON/text from an MCP tool CallToolResult."""
    if not hasattr(result, "content") or not result.content:
        return None
    text = result.content[0].text
    try:
        return json.loads(text)
    except Exception:
        return text

def extract_json_from_text(text: str) -> dict:
    """Extracts JSON object from text (handling markdown wrappers)."""
    match = re.search(r"({.*?})", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    try:
        return json.loads(text)
    except Exception:
        pass
    
    return {
        "score": 50,
        "match_level": "Medium",
        "matched_skills": [],
        "missing_skills": [],
        "rationale": "Failed to parse LLM response. Raw response: " + text[:200]
    }

def run_mock_llm_matching(resume_text: str, jd_text: str, profile_db_metadata: str) -> dict:
    """Fallback keyword-based mock matching in case no API keys are available."""
    jd_lower = jd_text.lower()
    resume_lower = resume_text.lower()
    
    tech_keywords = [
        "python", "javascript", "react", "aws", "docker", "kubernetes", 
        "sql", "java", "machine learning", "data scientist", "scrum master", 
        "cloud", "c++", "angular", "node.js", "django", "fastapi"
    ]
    
    matched = []
    missing = []
    
    for kw in tech_keywords:
        if kw in jd_lower:
            if kw in resume_lower:
                matched.append(kw.capitalize())
            else:
                missing.append(kw.capitalize())
                
    db_certs = []
    if profile_db_metadata:
        try:
            meta = json.loads(profile_db_metadata)
            db_certs = meta.get("certifications", [])
            matched.extend(db_certs)
        except Exception:
            pass
            
    total_jd = len(matched) + len(missing)
    if total_jd > 0:
        score = int((len(matched) / total_jd) * 100)
    else:
        score = 65
        
    level = "High" if score >= 80 else ("Medium" if score >= 50 else "Low")
        
    return {
        "score": score,
        "match_level": level,
        "matched_skills": list(set(matched)),
        "missing_skills": list(set(missing)),
        "rationale": f"Mock Matcher: Found {len(matched)} matching skills out of {total_jd} target keywords. Evaluated with database certifications: {db_certs}."
    }

def run_real_llm_matching(resume_text: str, jd_text: str, profile_db_metadata: str) -> dict:
    """Performs real LLM evaluation using OpenRouter or OpenAI."""
    from openai import OpenAI
    
    openrouter_key = os.getenv("OPENROUTER_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")
    
    prompt = f"""
You are an expert HR recruitment agent. Analyze the following candidate resume and match it against the Job Description.
Also consider the Candidate HR Database Profile metadata (background checks, certifications, expected salary) if available.

### Job Description:
{jd_text}

### Candidate Resume:
{resume_text}

### HR Database Profile Metadata:
{profile_db_metadata}

Provide your evaluation in JSON format with the following keys:
- score: an integer between 0 and 100 representing the overall match score.
- match_level: "High", "Medium", or "Low".
- matched_skills: a list of skills from the resume that match the job description.
- missing_skills: a list of skills from the job description that are missing from the resume.
- rationale: a 2-3 sentence explanation of your scoring.

Output ONLY valid JSON. Do not write anything else.
"""

    if openrouter_key:
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=openrouter_key
        )
        model = os.getenv("OPENROUTER_MODEL", "google/gemini-2.5-flash")
    elif openai_key:
        client = OpenAI(api_key=openai_key)
        model = "gpt-4o-mini"
    else:
        raise ValueError("No LLM key configured")
        
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return extract_json_from_text(response.choices[0].message.content)

## 7. Define LangGraph Node Functions

Here, we implement the step-by-step logic for our LangGraph nodes, which directly make tool/API calls to the active SSE server sessions.

In [8]:
async def read_resumes_node(state: AgentState) -> AgentState:
    """Discovers resumes in directory and parses them in parallel using MCP server."""
    fs_session = GLOBAL_FS_SESSION
    resumes_dir = state["resumes_dir"]
    
    print(f"[Node: Read Resumes] Discovering files in: {resumes_dir}")
    
    result = await fs_session.call_tool("list_directory", arguments={"path": resumes_dir})
    filenames = parse_mcp_result(result)
    
    if not isinstance(filenames, list):
        print(f"[Node: Read Resumes] Error listing files: {filenames}")
        state["resume_paths"] = []
        state["resume_contents"] = {}
        return state
        
    supported_files = [f for f in filenames if f.lower().endswith((".txt", ".pdf"))]
    abs_paths = [os.path.abspath(os.path.join(resumes_dir, f)) for f in supported_files]
    state["resume_paths"] = abs_paths
    
    if not abs_paths:
        print("[Node: Read Resumes] No supported files found (.txt or .pdf)")
        state["resume_contents"] = {}
        return state
        
    print(f"[Node: Read Resumes] Reading {len(abs_paths)} files in parallel...")
    batch_result = await fs_session.call_tool("batch_process", arguments={"paths": abs_paths})
    contents = parse_mcp_result(batch_result)
    
    state["resume_contents"] = contents if isinstance(contents, dict) else {}
    return state

async def fetch_profiles_node(state: AgentState) -> AgentState:
    """Queries secondary Database MCP Server to get candidate database profiles (Multi-MCP)."""
    db_session = GLOBAL_DB_SESSION
    profiles = {}
    
    print("[Node: Fetch DB Profiles] Querying Database MCP server...")
    
    for path in state["resume_paths"]:
        filename = os.path.basename(path)
        
        name_part = os.path.splitext(filename)[0]
        for term in ["_resume", "_cv", "-resume", "-cv", "resume", "cv"]:
            name_part = re.sub(term, "", name_part, flags=re.IGNORECASE)
        candidate_name = name_part.replace("_", " ").replace("-", " ").strip()
        
        result = await db_session.call_tool("get_candidate_profile", arguments={"name": candidate_name})
        profile_data = parse_mcp_result(result)
        
        if isinstance(profile_data, dict):
            profiles[path] = profile_data
        elif isinstance(profile_data, str) and profile_data.strip().startswith("{"):
            try:
                profiles[path] = json.loads(profile_data)
            except Exception:
                profiles[path] = {"error": profile_data}
        else:
            profiles[path] = {"error": profile_data or "Profile not found"}
            
    state["candidate_profiles"] = profiles
    return state

async def match_resumes_node(state: AgentState) -> AgentState:
    """Matches resume content against job description using Real or Mock LLM."""
    jd = state["job_description"]
    reports = []
    
    print("[Node: Match Resumes] Running candidate evaluation...")
    
    for path, content in state["resume_contents"].items():
        filename = os.path.basename(path)
        profile_info = state["candidate_profiles"].get(path, {})
        profile_str = json.dumps(profile_info, indent=2)
        
        print(f" -> Processing {filename}...")
        
        has_keys = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
        if has_keys:
            try:
                eval_res = run_real_llm_matching(content, jd, profile_str)
            except Exception as e:
                print(f"    [LLM Error] Real LLM matching failed: {e}. Falling back to keyword mock.")
                eval_res = run_mock_llm_matching(content, jd, profile_str)
        else:
            eval_res = run_mock_llm_matching(content, jd, profile_str)
            
        eval_res["file_path"] = path
        eval_res["filename"] = filename
        eval_res["candidate_name"] = profile_info.get("name", filename.replace("_", " ").split(".")[0].title())
        reports.append(eval_res)
        
    state["match_reports"] = reports
    return state

async def generate_report_node(state: AgentState) -> AgentState:
    """Ranks candidates, compiles a Markdown report, and writes it to disk via MCP server."""
    fs_session = GLOBAL_FS_SESSION
    reports = state["match_reports"]
    
    print("[Node: Generate Report] Ranking candidates and compiling report...")
    
    sorted_reports = sorted(reports, key=lambda x: x.get("score", 0), reverse=True)
    
    md = []
    md.append("# Candidate Matching & Ranking Report")
    md.append(f"**Generated At:** {time.strftime('%Y-%m-%d %H:%M:%S')}")
    md.append("\n## Job Description Summary")
    md.append(f"> {state['job_description'][:300]}...\n")
    
    md.append("## Executive Candidate Summary")
    md.append("| Rank | Candidate Name | Score | Fit Level | Matched Skills | Missing Skills |")
    md.append("| :--- | :--- | :---: | :---: | :--- | :--- |")
    
    for rank, r in enumerate(sorted_reports, 1):
        matched = ", ".join(r.get("matched_skills", [])) or "None"
        missing = ", ".join(r.get("missing_skills", [])) or "None"
        md.append(f"| {rank} | {r.get('candidate_name')} | **{r.get('score')}**/100 | {r.get('match_level')} | {matched} | {missing} |")
        
    md.append("\n## Detailed Evaluations")
    for rank, r in enumerate(sorted_reports, 1):
        md.append(f"### {rank}. {r.get('candidate_name')}")
        md.append(f"- **Resume File:** `{r.get('filename')}`")
        md.append(f"- **Overall Match Score:** **{r.get('score')}**/100")
        md.append(f"- **Fit Class:** {r.get('match_level')}")
        md.append(f"- **Matched Skills:** {', '.join(r.get('matched_skills', [])) or 'None'}")
        md.append(f"- **Missing Skills:** {', '.join(r.get('missing_skills', [])) or 'None'}")
        md.append(f"- **Recruiter Evaluation:** {r.get('rationale')}")
        
        profile = state["candidate_profiles"].get(r.get("file_path"), {})
        if profile and "error" not in profile:
            md.append(f"- **HR Database Integration:**")
            md.append(f"  - *Expected Salary:* {profile.get('expected_salary', 'N/A')}")
            md.append(f"  - *Background Check:* {profile.get('background_check', 'N/A')}")
            md.append(f"  - *Certifications:* {', '.join(profile.get('certifications', [])) or 'None'}")
            md.append(f"  - *DB Notes:* *{profile.get('notes', 'N/A')}*")
        md.append("")
        
    report_content = "\n".join(md)
    state["final_report"] = report_content
    
    report_path = os.path.abspath("./reports/latest_report.md")
    write_result = await fs_session.call_tool(
        "write_file", 
        arguments={"path": report_path, "content": report_content}
    )
    print(f"[Node: Generate Report] Saved report to {report_path}: {parse_mcp_result(write_result)}")
    return state

## 8. Assemble the LangGraph State Machine

We construct and compile the StateGraph using our step-by-step nodes.

In [9]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)
builder.add_node("read_resumes", read_resumes_node)
builder.add_node("fetch_profiles", fetch_profiles_node)
builder.add_node("match_resumes", match_resumes_node)
builder.add_node("generate_report", generate_report_node)

builder.add_edge(START, "read_resumes")
builder.add_edge("read_resumes", "fetch_profiles")
builder.add_edge("fetch_profiles", "match_resumes")
builder.add_edge("match_resumes", "generate_report")
builder.add_edge("generate_report", END)

workflow_app = builder.compile()
print("Workflow state machine compiled successfully!")

Workflow state machine compiled successfully!


## 9. Execute LangGraph Matching Pipeline over SSE

Finally, we connect to the active SSE transport sessions, set the global session variables for the graph nodes, define the target Job Description, and execute the matching agent!

In [ ]:
JOB_DESCRIPTION = """
Looking for a Senior Python Developer with extensive experience in LangChain, LangGraph, and cloud systems (AWS).
Required Skills: Python, LangGraph, LangChain, AWS, Docker.
Nice to have: SQL, Javascript.
"""

print("Connecting to external SSE servers...")
async with sse_client(FS_SERVER_URL) as (fs_read, fs_write), \
           sse_client(DB_SERVER_URL) as (db_read, db_write):
           
    async with ClientSession(fs_read, fs_write) as fs_session, \
               ClientSession(db_read, db_write) as db_session:
               
        await fs_session.initialize()
        await db_session.initialize()
        print("Handshake complete. Sessions set to node execution context.")
        
        # Bind sessions to the global variables used by node functions
        global GLOBAL_FS_SESSION, GLOBAL_DB_SESSION
        GLOBAL_FS_SESSION = fs_session
        GLOBAL_DB_SESSION = db_session
        
        # Initialize Graph State
        initial_state = {
            "job_description": JOB_DESCRIPTION,
            "resumes_dir": os.path.abspath(RESUMES_DIR),
            "resume_paths": [],
            "resume_contents": {},
            "candidate_profiles": {},
            "match_reports": [],
            "final_report": ""
        }
        
        print("\n--- Running LangGraph Agent Workflow ---")
        final_state = await workflow_app.ainvoke(initial_state)
        print("\n--- Workflow Complete! ---")
        
        print("\n================== GENERATED PIPELINE REPORT ==================")
        print(final_state["final_report"])
        print("===============================================================")

Connecting to external SSE servers...
Handshake complete. Sessions set to node execution context.

--- Running LangGraph Agent Workflow ---
[Node: Read Resumes] Discovering files in: d:\Github\MCP-integration\resumes
[Node: Read Resumes] No supported files found (.txt or .pdf)
[Node: Fetch DB Profiles] Querying Database MCP server...
[Node: Match Resumes] Running candidate evaluation...
[Node: Generate Report] Ranking candidates and compiling report...
[Node: Generate Report] Saved report to d:\Github\MCP-integration\reports\latest_report.md: Error: Access Denied: Path 'd:\Github\MCP-integration\reports\latest_report.md' (resolved to 'd:\Github\MCP-integration\reports\latest_report.md') is outside allowed directories: ['D:\\Github\\MCP-integration', 'D:\\Github\\MCP-integration\\resumes', 'D:\\Github\\MCP-integration\\reports']

--- Workflow Complete! ---

================== GENERATED PIPELINE REPORT ==================
# Candidate Matching & Ranking Report
**Generated At:** 2026-08-24 